# Estimated JAR Build Cost

This notebook derives the build-time estimates reported for the longitudinal study from the sampled build measurements and commit histories.


In [ ]:
from pathlib import Path

import pandas as pd

BUILD_TIMES_PATH = Path("library-build-times.csv")
HISTORY_DIR = Path("data")


In [ ]:
build_times = pd.read_csv(BUILD_TIMES_PATH)
build_times["elapsed_seconds"] = pd.to_numeric(build_times["elapsed_seconds"], errors="raise")
build_times["library"] = build_times["repo_name"].str.replace("__", "/", regex=False)

median_build_times = (
    build_times.groupby(["repo_name", "library"], as_index=False)
    .agg(
        build_attempts=("commit", "size"),
        median_build_seconds=("elapsed_seconds", "median"),
    )
)


In [ ]:
history_rows = []
for path in sorted(HISTORY_DIR.glob("*-commits.csv")):
    history = pd.read_csv(path, usecols=["library", "commit_url"], low_memory=False)
    library = history["library"].dropna().iat[0]
    url_parts = history["commit_url"].dropna().astype(str).str.split("/")
    repository_slug = (url_parts.str[3] + "/" + url_parts.str[4]).mode().iat[0]
    history_rows.append(
        {
            "library": library,
            "repo_name": repository_slug.replace("/", "__"),
            "commits": len(history),
        }
    )

estimated_build_cost = (
    pd.DataFrame(history_rows)
    .merge(median_build_times.drop(columns="library"), on="repo_name", validate="many_to_one")
    .assign(estimated_total_build_hours=lambda frame: frame["commits"] * frame["median_build_seconds"] / 3600)
    .sort_values("estimated_total_build_hours", ascending=False)
    .reset_index(drop=True)
)

paper_build_cost_table = estimated_build_cost[[
    "library",
    "commits",
    "median_build_seconds",
    "estimated_total_build_hours",
]]
display(paper_build_cost_table.round({"median_build_seconds": 2, "estimated_total_build_hours": 1}))
print(f"Estimated total build cost: {estimated_build_cost['estimated_total_build_hours'].sum():.1f} hours")
